# 🤖 Notebook 03 — Model Training

**Project:** ASD Detection in Children using Machine Learning  

---

## 🎯 Objectives

In this notebook we will:
1. Build a reproducible sklearn **Pipeline** (preprocessor + classifier)
2. Train **5 different classifiers** on the ASD dataset
3. Apply **5-fold cross-validation** to get robust estimates
4. Compare training and validation accuracy
5. Discuss the strengths and weaknesses of each model

---

## 📌 Why Multiple Models?

No single algorithm dominates all datasets. By training multiple models we:
- Get an unbiased view of which algorithm fits this data best
- Identify potential for ensembling
- Provide a rigorous, reproducible comparison

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, os.path.join('..'))  # project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from ml_models.preprocessing import clean_raw_dataframe, build_preprocessor, FEATURE_COLS, TARGET_COL

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'text.color': '#e2eaf5', 'figure.dpi': 110,
})

RANDOM_STATE = 42
print('✅ Imports complete')

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('../data/data_csv.csv')
df = clean_raw_dataframe(df)

available = [c for c in FEATURE_COLS if c in df.columns]
X = df[available]
y = df[TARGET_COL].fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Features used: {len(available)}')
print(f'Class balance: {y_train.value_counts().to_dict()}')

## 2. Define Models

Each model is wrapped in a `Pipeline` with a preprocessor. This ensures that:
1. Scaling is fit **only on training data** (no data leakage)
2. The same transformations are applied consistently at inference

In [ ]:
def make_pipeline(clf):
    return Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', clf)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
    ),
    'Decision Tree': make_pipeline(
        DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=None,
                               min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
    ),
    'KNN': make_pipeline(
        KNeighborsClassifier(n_neighbors=5, metric='euclidean')
    ),
    'SVM': make_pipeline(
        SVC(kernel='rbf', C=1.0, probability=True, random_state=RANDOM_STATE)
    ),
}

print('Models defined:', list(models.keys()))

## 3. Model Descriptions

### 📘 Logistic Regression
A linear classifier that models the probability of a class using a sigmoid function. Fast, interpretable, and a strong baseline. Works best when the decision boundary is approximately linear.

### 🌳 Decision Tree
Splits the feature space into rectangular regions using information gain or Gini impurity. Highly interpretable (can be visualised as a tree), but prone to overfitting without depth constraints.

### 🌲 Random Forest
An ensemble of decision trees trained on bootstrap samples with random feature subsets. Reduces variance by averaging predictions. Often the best performer on tabular clinical data.

### 🔵 K-Nearest Neighbors (KNN)
Classifies a sample by majority vote of its K nearest training examples. Non-parametric, no training phase, but sensitive to feature scale and slow at inference on large datasets.

### ⚡ Support Vector Machine (SVM)
Finds the maximum-margin hyperplane separating classes. With an RBF kernel, it can handle non-linear boundaries. Powerful but computationally expensive.

## 4. Train & Cross-Validate

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

print(f"{'Model':<28} {'Train Acc':>10} {'Test Acc':>10} {'CV Mean':>10} {'CV Std':>8}")
print('─' * 70)

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc  = accuracy_score(y_test, pipe.predict(X_test))
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    
    results[name] = {
        'pipeline':  pipe,
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'cv_mean':   cv_scores.mean(),
        'cv_std':    cv_scores.std(),
        'cv_scores': cv_scores,
    }
    
    print(f"{name:<28} {train_acc:>10.4f} {test_acc:>10.4f} {cv_scores.mean():>10.4f} {cv_scores.std():>8.4f}")

print('─' * 70)

## 5. Visualise Training Results

In [ ]:
names  = list(results.keys())
train_accs = [results[n]['train_acc'] * 100 for n in names]
test_accs  = [results[n]['test_acc']  * 100 for n in names]
cv_means   = [results[n]['cv_mean']   * 100 for n in names]
cv_stds    = [results[n]['cv_std']    * 100 for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: train vs test
x = np.arange(len(names))
w = 0.35
axes[0].bar(x - w/2, train_accs, w, color='#6366f1', label='Train', edgecolor='#0f172a')
axes[0].bar(x + w/2, test_accs,  w, color='#10b981', label='Test',  edgecolor='#0f172a')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylim(70, 105)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Train vs. Test Accuracy')
axes[0].legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white')

# CV with error bars
colors = ['#6366f1', '#f59e0b', '#10b981', '#3b82f6', '#ec4899']
axes[1].bar(x, cv_means, color=colors, edgecolor='#0f172a')
axes[1].errorbar(x, cv_means, yerr=cv_stds, fmt='none', color='white', capsize=5, capthick=2)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    axes[1].text(i, m + s + 0.5, f'{m:.1f}%', ha='center', color='white', fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylim(70, 105)
axes[1].set_ylabel('CV Accuracy (%)')
axes[1].set_title('5-Fold Cross-Validation Accuracy')

plt.tight_layout()
plt.show()

## 6. Save All Trained Models

In [ ]:
os.makedirs('../ml_models', exist_ok=True)
for name, res in results.items():
    safe_name = name.lower().replace(' ', '_')
    path = f'../ml_models/{safe_name}.pkl'
    joblib.dump(res['pipeline'], path)
    print(f'  ✅ Saved: {path}')

print('\nAll models saved.')

---

## ➡️ Next: Notebook 04 — Evaluation & Model Selection